# Block Ciphers

In [ ]:
from random import seed, randbytes, randint
from itertools import product

import numpy as np


In [ ]:
# %matplotlib widget
import matplotlib.pyplot as plt

Unlike **stream ciphers**, which encrypt data one bit at a time, **block ciphers** encrypt data in fixed-size blocks. For example, **AES** encrypts blocks of **128 bits**.

Since a block cipher can only encrypt a single block under a fixed key, several **modes of operation** have been designed. These modes allow block ciphers to be used repeatedly and securely on longer messages.

Moreover, block ciphers can also be used as building blocks in other cryptographic protocols, such as:
- universal hash functions
- pseudo-random number generators

![Caption](images/block_ciphers.png)

## Advanced Encryption Standard (AES)



**AES** is a specification for **symmetric-key encryption** established by **NIST** in **2001** and later adopted by the **U.S. government**.

The AES standard includes three block ciphers derived from the Rijndael cipher family.  
Each cipher has a fixed **128-bit block size**, but supports different key sizes:

| Key length | Number of rounds |
|----------:|-----------------:|
| 128 bits  | 10 rounds |
| 192 bits  | 12 rounds |
| 256 bits  | 14 rounds |

AES is composed of two main parts:

- a **key expansion block**
- a **data path**

The data path can be viewed as an **iterated block cipher**, where each iteration is called a **round**.

The number of rounds depends on the key length.

**References**: [FIPS PUB 197](https://csrc.nist.gov/pubs/fips/197/final), *Advanced Encryption Standard (AES)*, National Institute of Standards and Technology, U.S. Department of Commerce, November 2001.


## The Rijndael Block Cipher

Cryptographic algorithm designed by Joan Daemen and Vincent Rijmen (2) that became the basis of AES (AES is the standardized 128-bit-block version of Rijndael) .

The **round transformation** of Rijndael/AES is composed of three distinct invertible transformations, called **layers**.

In these layers, every bit of the state is treated in a similar way.

The three layers are:
1) <span style="color:#E57373"><strong>Non-linear layer</strong></span>: provides **confusion**
2) <span style="color:#2E8B57"><strong>Linear mixing layer</strong></span>: provides **diffusion**
3) <span style="color:#1E90FF"><strong>Key addition layer</strong></span>: inserts the key into the datapath

**References**: 2. Joan Daemen and Vincent Rijmen, [*The Design of Rijndael: AES — The Advanced Encryption Standard*](https://link.springer.com/book/10.1007/978-3-662-04722-4), Springer-Verlag, 2002.

![Caption](images/Rijndael_Block_Cipher.png)

The **internal state** of AES is a sequence of **128 bits** (**16 bytes**), represented as a **4×4 matrix of bytes**.

In [ ]:
block_size = 128 // 8
state = randbytes(block_size)

In [ ]:
def print_state(state):
    print(''.join([f'{byte:02x}' for j, byte in enumerate(state)]))

def show_state(state, ax=None, title=None):
    mat = np.array(list(state), dtype=np.uint8).reshape(-1, 4).T
    shape = mat.shape

    if ax is None:
        fig, ax = plt.subplots(figsize=(shape[0]/2, shape[1]/2))

    ax.matshow(mat, vmin=0, vmax=255, cmap='gray')
    ax.set(xticks=[], yticks=[])
    if title is not None:
        ax.set(title=title)

    for i, j in product(range(shape[0]), range(shape[1])):
        color = 'w' if mat[i, j] < 128 else 'k'
        text = ax.text(j, i, f'{mat[i, j]:02x}',
                       ha="center", va="center", color=color)

In [ ]:
show_state(state)
plt.show()

### <span style="color:#1E90FF"><strong>AddRoundKey</strong></span>

Internally, Rijndael represents the 128-bit state as a 4 × 4 matrix of bytes.

The **AddRoundKey** operation consists of a bitwise XOR between the current state and the round key derived from the cipher key:

$
Bj = Aj \oplus Kj
$

This step is the **key addition layer**, because it inserts the secret key material into the encryption process.

|  | Hex | Binary |
|:----------|:---:|:--------:|
| State byte | `53 53 53 53 53 53 53 53 53 53 53 53 53 53 53 53` | `01010011`$\times 16$ |
| Round key byte | `2f 2f 2f 2f 2f 2f 2f 2f 2f 2f 2f 2f 2f 2f 2f 2f` | `00101111`$\times 16$ |
| Result byte | `7c 7c 7c 7c 7c 7c 7c 7c 7c 7c 7c 7c 7c 7c 7c 7c` | `01111100`$\times 16$ |

In [ ]:
def add_round_key(state: bytes, round_key: bytes) -> bytes:
    # CODE HERE
    return state

Tutorial **Type hinting**: is a way of showing the expected data types of function inputs and outputs.

For example:

- `state: bytes` means `state` should be a `bytes` value.
- `round_key: bytes` means `round_key` should be a `bytes` value.
- `-> bytes` means the function should return a `bytes` value.

Type hinting is useful because it makes code easier to read, helps find mistakes earlier, and improves editor support such as autocomplete.

In [ ]:
round_key = randbytes(block_size)
state_input = randbytes(block_size)
state_output = add_round_key(state_input, round_key)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(8, 2))
show_state(round_key, ax=ax1, title='key')
show_state(state_input, ax=ax2, title='input')
show_state(state_output, ax=ax3, title='output')
plt.show()

### <span style="color:#E57373"><strong>Byte substitution</strong></span>:
is a non-linear byte substitution that operates independently on each byte of the State using a substitution table (S-box).



![Caption](images/byte_substitution.png)

In [ ]:
sbox = bytes.fromhex(
        # 0 1 2 3 4 5 6 7 8 9 a b c d e f
        '637c777bf26b6fc53001672bfed7ab76' # 0
        'ca82c97dfa5947f0add4a2af9ca472c0' # 1
        'b7fd9326363ff7cc34a5e5f171d83115' # 2
        '04c723c31896059a071280e2eb27b275' # 3
        '09832c1a1b6e5aa0523bd6b329e32f84' # 4
        '53d100ed20fcb15b6acbbe394a4c58cf' # 5
        'd0efaafb434d338545f9027f503c9fa8' # 6
        '51a3408f929d38f5bcb6da2110fff3d2' # 7
        'cd0c13ec5f974417c4a77e3d645d1973' # 8
        '60814fdc222a908846eeb814de5e0bdb' # 9
        'e0323a0a4906245cc2d3ac629195e479' # a
        'e7c8376d8dd54ea96c56f4ea657aae08' # b
        'ba78252e1ca6b4c6e8dd741f4bbd8b8a' # c
        '703eb5664803f60e613557b986c11d9e' # d
        'e1f8981169d98e949b1e87e9ce5528df' # e
        '8ca1890dbfe6426841992d0fb054bb16' # f
    )

In [ ]:
def byte_substitution(state: bytes) -> bytes:
    # CODE HERE
    return state


def inverse_byte_substitution(state: bytes) -> bytes:
    # CODE HERE
    return state

In [ ]:
state_inp = randbytes(block_size)
state_enc = byte_substitution(state_inp)
state_dec = inverse_byte_substitution(state_enc)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(8, 2))
show_state(state_inp, ax=ax1, title='input')
show_state(state_enc, ax=ax2, title='encrypted')
show_state(state_dec, ax=ax3, title='decrypted')

plt.show()

### <span style="color:#2E8B57"><strong>Shift Rows</strong></span>:

The rows of the state are cyclically shifted over different numbers of bytes (offsets). Therefore, it simply consists in a bytes permutation of the state.

![Caption](images/shift_rows.png)

In [ ]:
def shift_rows(state: bytes) -> bytes:
    # CODE HERE
    return state

def inverse_shift_rows(state: bytes) -> bytes:
    # CODE HERE
    return state

In [ ]:
state_inp = randbytes(block_size)
state_shifted = shift_rows(state_inp)
state_ = inverse_shift_rows(state_shifted)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(8, 2))
show_state(state_inp, ax=ax1, title='input')
show_state(state_shifted, ax=ax2, title='shifted')
show_state(state_, ax=ax3, title='inverse')

plt.show()

### <span style="color:#2E8B57"><strong>Mix Columns</strong></span>:
It's the AES operation that gives diffusion: it mixes the bytes inside each column of the state, so that one input byte affects several output bytes.
It operates on the state column-by-column by computing a matrix-vector multiplication in which each element is in $GF(2^8)$,

When two bytes are multiplied as polynomials, the product may have degree greater than 7. Therefore, AES uses the following irreducible polynomial as fixed rule to reduce the product back into one byte. 

$$
P(x)=x^8+x^4+x^3+x+1.
$$



![Caption](images/mix_columns.png)

**Note** that there are **only three values to be multiplied** to a byte: 01, 02, and 03. Therefore, you do not need to implement the multiplication between two generic bytes but only the multiplication of a byte by those three values: **01∙𝐴**, **02∙𝐴**, and **03∙𝐴**.


**Implementation tricks**:

In MixColumns, bytes are only multiplied by the constants $01$, $02$, and $03$. Therefore, AES does not need generic multiplication between two arbitrary bytes in $GF(2^8)$.

Multiplying by $01$ leaves the byte unchanged:

$$
01 \cdot A = A.
$$

Multiplying by $02$ means multiplying the byte polynomial by $x$. In binary, this is almost the same as a left shift by one bit.

If the original most significant bit of $A$ is $0$, the left shift does not overflow, so:

$$
02 \cdot A = A \ll 1.
$$

If the original most significant bit of $A$ is $1$, the left shift produces a term of degree $8$, which does not fit in one byte. Therefore, AES reduces the result modulo

$$
P(x)=x^8+x^4+x^3+x+1.
$$

In practice, this reduction is implemented by XOR-ing with `0x1B`.

So the implementation rule is:

$$
02 \cdot A =
\begin{cases}
A \ll 1, & \text{if the MSB of } A \text{ is } 0,\\
(A \ll 1) \oplus \texttt{0x1B}, & \text{if the MSB of } A \text{ is } 1.
\end{cases}
$$

Multiplication by $03$ is computed using the fact that $03 = 02 \oplus 01$:

$$
03 \cdot A = (02 \cdot A) \oplus A.
$$

In [ ]:
def mix_columns(state: bytes) -> bytes:
    # CODE HERE
    return state


In [ ]:
state_input = bytes.fromhex('6347a2f0f20a225c010101012d26314c')
state_enc = mix_columns(state_input)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 2))
show_state(state_input, ax=ax1, title='input')
show_state(state_enc, ax=ax2, title='encrypted')

plt.show()

|  | Input | Encrypted |
|---:|---|---|
| 1 | `63 f2 01 2d` | `5d 9f 01 4d` | 
| 2 | `47 0a 01 26` | `e0 dc 01 7e` | 
| 3 | `a2 22 01 31` | `70 58 01 bd` | 
| 4 | `f0 5c 01 4c` | `bb 9d 01 f8` | 


## Homework:

In [ ]:
def inverse_mix_columns(state: bytes) -> bytes:
    # CODE HERE
    return state

In [ ]:
state_input = bytes.fromhex('6347a2f0f20a225c010101012d26314c')
state_enc = mix_columns(state_input)
state_dec = mix_columns(state_enc)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(8, 2))
show_state(state_input, ax=ax1, title='input')
show_state(state_enc, ax=ax2, title='encrypted')
show_state(state_dec, ax=ax3, title='decrypted')

plt.show()

|  | Input | Encrypted | Decrypted |
|---:|---|---|---|
| 1 | `63 f2 01 2d` | `5d 9f 01 4d` | `63 f2 01 2d` |
| 2 | `47 0a 01 26` | `e0 dc 01 7e` | `47 0a 01 26` |
| 3 | `a2 22 01 31` | `70 58 01 bd` | `a2 22 01 31` |
| 4 | `f0 5c 01 4c` | `bb 9d 01 f8` | `f0 5c 01 4c` |


### Round

In [ ]:
def round(state: bytes , round_keys: bytes) -> bytes:
    # CODE HERE
    return state


| Row | Input | Encrypted |
|---:|---|---|
| 1 | `19 a0 9a e9` | `a4 68 6b 02` | 
| 2 | `3d f4 c6 f8` | `9c 9f 5b 6a` | 
| 3 | `e3 e2 8d 48` | `7f 35 ea 50` | 
| 4 | `be 2b 2a 08` | `f2 2b 43 49` | 

In [ ]:
round_key = bytes.fromhex('a0fafe1788542cb123a339392a6c7605')
state_input = bytes.fromhex('193de3bea0f4e22b9ac68d2ae9f84808')
state_enc = round(state_input, round_key)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 2))
show_state(state_input, ax=ax1, title='input')
show_state(state_enc, ax=ax2, title='encrypted')

plt.show()

## Homework:

In [ ]:
def inverse_round(state: bytes, round_keys: bytes) -> bytes:
    # CODE HERE
    return state

|  | Input | Encrypted | Decrypted |
|---:|---|---|---|
| 1 | `19 a0 9a e9` | `a4 68 6b 02` | `19 a0 9a e9` |
| 2 | `3d f4 c6 f8` | `9c 9f 5b 6a` | `3d f4 c6 f8` |
| 3 | `e3 e2 8d 48` | `7f 35 ea 50` | `e3 e2 8d 48` |
| 4 | `be 2b 2a 08` | `f2 2b 43 49` | `be 2b 2a 08` |

### <span style="color:grey"><strong>Key Expansion</strong></span>:

AES does not use the original cipher key directly in every round. Instead, it **expands the cipher key into several round keys**.

For AES-128, the cipher key has length $128$ bits, i.e. $16$ bytes or $4$ words. Therefore, $N_k=4$. Since AES-128 has $N_r=10$ rounds, the algorithm generates $N_r+1=11$ round keys:

$$
K_0, K_1, \dots, K_{10}.
$$

The first round key is the original cipher key:

$$
K_0 = K.
$$

Each round key contains $4$ words, i.e. $16$ bytes. The remaining round keys are generated word by word from the previous key material. For AES-128, the expansion is organized into $10$ stages, where each stage generates $N_k=4$ new words, corresponding to the next round key.


![Caption](images/key_expansion.png)

very stage implements a very similar transformation. The only difference is in the round coefficient (RC) that is different for each round.

![Caption](images/key_expansion_2.png)

In [ ]:
def key_expansion(key: bytes) -> bytes:
    # CODE HERE
    return state

In [ ]:
def print_round_keys(round_keys):
    for i, round_key in enumerate(round_keys):
        print(f'{i:2d}', end=' ')
        for j, byte in enumerate(round_key):
            if j % 4 == 0:
                print('', end=' ')
            print(f'{byte:02x}', end='')
        print()

In [ ]:
# key = randbytes(AES.key_sizes[0])
# print(key)
key = bytes.fromhex('2b7e151628aed2a6abf7158809cf4f3c')
round_keys = key_expansion(key)
print_round_keys(round_keys)


| Round | Word 0 | Word 1 | Word 2 | Word 3 |
|---:|---|---|---|---|
| 0 | `2b7e1516` | `28aed2a6` | `abf71588` | `09cf4f3c` |
| 1 | `a0fafe17` | `88542cb1` | `23a33939` | `2a6c7605` |
| 2 | `f2c295f2` | `7a96b943` | `5935807a` | `7359f67f` |
| 3 | `3d80477d` | `4716fe3e` | `1e237e44` | `6d7a883b` |
| 4 | `ef44a541` | `a8525b7f` | `b671253b` | `db0bad00` |
| 5 | `d4d1c6f8` | `7c839d87` | `caf2b8bc` | `11f915bc` |
| 6 | `6d88a37a` | `110b3efd` | `dbf98641` | `ca0093fd` |
| 7 | `4e54f70e` | `5f5fc9f3` | `84a64fb2` | `4ea6dc4f` |
| 8 | `ead27321` | `b58dbad2` | `312bf560` | `7f8d292f` |
| 9 | `ac7766f3` | `19fadc21` | `28d12941` | `575c006e` |
| 10 | `d014f9a8` | `c9ee2589` | `e13f0cc8` | `b6630ca6` |


# Homework:

Implement the __AES__ class using the defining as methods the functions we discussed earlier. Complete any missing methods, and provide **encrypt** and **decrypt** methods.

In [ ]:
class AES:
    # CODE HERE
    return

In [ ]:
key = bytes.fromhex('00000000000000000000000000000000')
plaintext = bytes.fromhex('00000000000000000000000000000000')

aes = AES(key)
ciphertext = aes.encrypt(plaintext)
decrypted = aes.decrypt(ciphertext)

print(' plaintext:', ''.join([f'{byte:02x}' for byte in plaintext]))
print('ciphertext:', ''.join([f'{byte:02x}' for byte in ciphertext]))
print(' decrypted:', ''.join([f'{byte:02x}' for byte in decrypted]))

|  |  |
|---|---|
| Plaintext | `00000000000000000000000000000000` |
| Ciphertext | `66e94bd4ef8a2c3b884cfa59ca342b2e` |
| Decrypted plaintext | `00000000000000000000000000000000` |

In [ ]:
key = bytes.fromhex('2b7e151628aed2a6abf7158809cf4f3c')
plaintext = bytes.fromhex('3243f6a8885a308d313198a2e0370734')

aes = AES(key)
ciphertext = aes.encrypt(plaintext)
decrypted = aes.decrypt(ciphertext)

print(' plaintext:', ''.join([f'{byte:02x}' for byte in plaintext]))
print('ciphertext:', ''.join([f'{byte:02x}' for byte in ciphertext]))
print(' decrypted:', ''.join([f'{byte:02x}' for byte in decrypted]))


|  |  |
|---|---|
| Plaintext, hex | `3243f6a8885a308d313198a2e0370734` |
| Ciphertext, hex | `3925841d02dc09fbdc118597196a0b32` |
| Decrypted plaintext, hex | `3243f6a8885a308d313198a2e0370734` |